# 🌍 NASA EONET — Earth Natural Events Dashboard

An interactive dashboard powered by the [NASA EONET v3 API](https://eonet.gsfc.nasa.gov/docs/v3), tracking wildfires, storms, volcanic eruptions, icebergs, and more — in real time.

In [79]:
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from IPython.display import display, HTML

## 1 — Fetch data from the EONET API

We pull **all events from the last 365 days** (open + closed) so we have a rich dataset to explore.

In [80]:
BASE = "https://eonet.gsfc.nasa.gov/api/v3"

# Fetch events (last 365 days, open + closed)
events_resp = requests.get(f"{BASE}/events", params={"status": "all", "days": 365, "limit": 1000})
events_resp.raise_for_status()
events_json = events_resp.json()

# Fetch categories for reference
categories_resp = requests.get(f"{BASE}/categories")
categories_resp.raise_for_status()
categories_json = categories_resp.json()

print(f"Fetched {len(events_json['events'])} events")
print(f"Available categories: {[c['title'] for c in categories_json['categories']]}")

Fetched 1000 events
Available categories: ['Drought', 'Dust and Haze', 'Earthquakes', 'Floods', 'Landslides', 'Manmade', 'Sea and Lake Ice', 'Severe Storms', 'Snow', 'Temperature Extremes', 'Volcanoes', 'Water Color', 'Wildfires']


## 2 — Parse events into a DataFrame

In [81]:
rows = []
for ev in events_json["events"]:
    cat = ev["categories"][0]["title"] if ev["categories"] else "Unknown"
    cat_id = ev["categories"][0]["id"] if ev["categories"] else "unknown"
    is_open = ev["closed"] is None
    for geom in ev["geometry"]:
        coords = geom["coordinates"]
        # Handle Point geometry [lon, lat]
        if isinstance(coords[0], (int, float)):
            lon, lat = coords[0], coords[1]
        else:
            # Polygon — take centroid of first ring
            ring = coords[0]
            lon = sum(p[0] for p in ring) / len(ring)
            lat = sum(p[1] for p in ring) / len(ring)
        rows.append({
            "id": ev["id"],
            "title": ev["title"],
            "category": cat,
            "category_id": cat_id,
            "date": geom["date"],
            "lat": lat,
            "lon": lon,
            "mag_value": geom.get("magnitudeValue"),
            "mag_unit": geom.get("magnitudeUnit"),
            "is_open": is_open,
        })

df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M").astype(str)
df["week"] = df["date"].dt.to_period("W").apply(lambda r: r.start_time)

print(f"{len(df)} geometry data-points from {df['id'].nunique()} unique events")
df.head()

1103 geometry data-points from 1000 unique events


/var/folders/8t/py7199996lzcf46ws17_9lp80000gn/T/ipykernel_30155/3554347978.py:31: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"] = df["date"].dt.to_period("M").astype(str)
/var/folders/8t/py7199996lzcf46ws17_9lp80000gn/T/ipykernel_30155/3554347978.py:32: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["week"] = df["date"].dt.to_period("W").apply(lambda r: r.start_time)


,id,title,category,category_id,date,lat,lon,mag_value,mag_unit,is_open,month,week
0,EONET_19376,Wildfire in Australia 1028218,Wildfires,wildfires,2026-04-06 19:00:00+00:00,-15.731795,128.369257,5263.0,hectare,False,2026-04,2026-04-06
1,EONET_19377,Wildfire in Mongolia 1028219,Wildfires,wildfires,2026-04-06 19:00:00+00:00,49.287566,115.782819,5134.0,hectare,False,2026-04,2026-04-06
2,EONET_19349,"Holly Springs IU 81-1 83-1 RX Prescribed Fire,...",Wildfires,wildfires,2026-04-06 09:15:00+00:00,34.497590,-89.382263,2113.0,acres,True,2026-04,2026-04-06
3,EONET_19351,"EASTER PASTURE Wildfire, Hendry, Florida",Wildfires,wildfires,2026-04-05 22:32:00+00:00,26.471389,-81.026944,562.0,acres,True,2026-04,2026-03-30
4,EONET_19329,Tropical Cyclone Vaianu,Severe Storms,severeStorms,2026-04-05 00:00:00+00:00,-13.800000,171.800000,40.0,kts,True,2026-04,2026-03-30


## 3 — Summary statistics

In [82]:
n_events = df["id"].nunique()
n_open = df.loc[df["is_open"], "id"].nunique()
n_closed = n_events - n_open
n_categories = df["category"].nunique()
date_range = f"{df['date'].min():%Y-%m-%d}  →  {df['date'].max():%Y-%m-%d}"

summary_html = f"""
<div style="display:flex; gap:24px; flex-wrap:wrap; margin:12px 0;">
  <div style="background:#1a1a2e; color:#e0e0e0; padding:20px 28px; border-radius:12px; min-width:150px; text-align:center;">
    <div style="font-size:36px; font-weight:700; color:#00d2ff;">{n_events}</div>
    <div style="font-size:13px; margin-top:4px;">Total Events</div>
  </div>
  <div style="background:#1a1a2e; color:#e0e0e0; padding:20px 28px; border-radius:12px; min-width:150px; text-align:center;">
    <div style="font-size:36px; font-weight:700; color:#ff6b6b;">{n_open}</div>
    <div style="font-size:13px; margin-top:4px;">Currently Open</div>
  </div>
  <div style="background:#1a1a2e; color:#e0e0e0; padding:20px 28px; border-radius:12px; min-width:150px; text-align:center;">
    <div style="font-size:36px; font-weight:700; color:#51cf66;">{n_closed}</div>
    <div style="font-size:13px; margin-top:4px;">Closed</div>
  </div>
  <div style="background:#1a1a2e; color:#e0e0e0; padding:20px 28px; border-radius:12px; min-width:150px; text-align:center;">
    <div style="font-size:36px; font-weight:700; color:#ffd43b;">{n_categories}</div>
    <div style="font-size:13px; margin-top:4px;">Categories</div>
  </div>
</div>
<p style="color:#888; font-size:12px;">Date range: {date_range}</p>
"""
display(HTML(summary_html))

## 4 — Interactive world map of natural events

In [83]:
# Take the latest geometry point per event for the map
latest = df.sort_values("date").drop_duplicates(subset="id", keep="last").copy()

# Category color map
cat_colors = {
    "Wildfires": "#ff4500",
    "Volcanoes": "#ff6347",
    "Severe Storms": "#6a5acd",
    "Sea and Lake Ice": "#00bfff",
    "Icebergs": "#87ceeb",
    "Floods": "#1e90ff",
    "Earthquakes": "#daa520",
    "Drought": "#cd853f",
    "Landslides": "#8b4513",
    "Snow": "#fffafa",
    "Temperature Extremes": "#ff1493",
    "Water Color": "#20b2aa",
    "Dust and Haze": "#d2b48c",
    "Manmade": "#808080",
}
latest["color"] = latest["category"].map(cat_colors).fillna("#cccccc")
latest["status"] = latest["is_open"].map({True: "🔴 Open", False: "✅ Closed"})

fig_map = px.scatter_geo(
    latest,
    lat="lat",
    lon="lon",
    color="category",
    hover_name="title",
    hover_data={"date": True, "status": True, "lat": ":.2f", "lon": ":.2f", "category": False},
    color_discrete_map=cat_colors,
    title="🌎 Active & Recent Natural Events Around the Globe",
    size_max=8,
)
fig_map.update_traces(marker=dict(size=7, opacity=0.85, line=dict(width=0.3, color="white")))
fig_map.update_geos(
    showcountries=True, countrycolor="#444",
    showcoastlines=True, coastlinecolor="#555",
    showland=True, landcolor="#1a1a2e",
    showocean=True, oceancolor="#0d1b2a",
    showlakes=True, lakecolor="#0d1b2a",
    projection_type="natural earth",
)
fig_map.update_layout(
    template="plotly_dark",
    height=550,
    margin=dict(l=0, r=0, t=50, b=0),
    legend=dict(title="Category", font_size=11),
)
fig_map.show()

## 5 — Events by category (treemap + bar)

In [84]:
cat_counts = (
    latest.groupby("category")["id"]
    .nunique()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

fig_tree = px.treemap(
    cat_counts,
    path=["category"],
    values="count",
    color="count",
    color_continuous_scale="YlOrRd",
    title="Event Distribution by Category (Treemap)",
)
fig_tree.update_layout(template="plotly_dark", height=420, margin=dict(t=50, l=10, r=10, b=10))
fig_tree.show()

In [85]:
fig_bar = px.bar(
    cat_counts,
    x="count",
    y="category",
    orientation="h",
    color="count",
    color_continuous_scale="Viridis",
    title="Number of Events per Category",
    text="count",
)
fig_bar.update_layout(template="plotly_dark", height=400, yaxis=dict(categoryorder="total ascending"), showlegend=False)
fig_bar.update_traces(textposition="outside")
fig_bar.show()

## 6 — Timeline: events per week

In [86]:
timeline = (
    df.drop_duplicates(subset=["id", "week"])
    .groupby(["week", "category"])["id"]
    .nunique()
    .reset_index(name="events")
)

fig_timeline = px.area(
    timeline,
    x="week",
    y="events",
    color="category",
    color_discrete_map=cat_colors,
    title="Weekly Event Activity by Category",
    labels={"week": "Week", "events": "Events"},
)
fig_timeline.update_layout(template="plotly_dark", height=420, xaxis_title="", legend_title="Category")
fig_timeline.show()

## 7 — Open vs Closed events over time

In [87]:
status_time = (
    df.drop_duplicates(subset=["id", "month"])
    .assign(status=lambda d: d["is_open"].map({True: "Open", False: "Closed"}))
    .groupby(["month", "status"])["id"]
    .nunique()
    .reset_index(name="events")
)

fig_status = px.bar(
    status_time,
    x="month",
    y="events",
    color="status",
    barmode="group",
    color_discrete_map={"Open": "#ff6b6b", "Closed": "#51cf66"},
    title="Monthly Events: Open vs Closed",
)
fig_status.update_layout(template="plotly_dark", height=380, xaxis_title="", yaxis_title="Events")
fig_status.show()

## 8 — Top 15 longest-running open events

In [88]:
open_events = df[df["is_open"]].copy()

if not open_events.empty:
    duration = (
        open_events.groupby(["id", "title", "category"])
        .agg(first_seen=("date", "min"), last_seen=("date", "max"), data_points=("date", "count"))
        .reset_index()
    )
    duration["days_active"] = (pd.Timestamp.now(tz="UTC") - duration["first_seen"]).dt.days
    top_open = duration.nlargest(15, "days_active")

    fig_dur = px.bar(
        top_open,
        y="title",
        x="days_active",
        color="category",
        color_discrete_map=cat_colors,
        orientation="h",
        title="Top 15 Longest-Running Open Events (days active)",
        text="days_active",
        hover_data={"data_points": True, "first_seen": True},
    )
    fig_dur.update_layout(
        template="plotly_dark", height=480,
        yaxis=dict(categoryorder="total ascending", title=""),
        xaxis_title="Days Active",
    )
    fig_dur.update_traces(textposition="outside")
    fig_dur.show()
else:
    print("No open events in the dataset.")

## 9 — Magnitude analysis (where available)

In [89]:
mag_df = df.dropna(subset=["mag_value"]).copy()

if not mag_df.empty:
    mag_df["mag_label"] = mag_df["mag_value"].astype(str) + " " + mag_df["mag_unit"].fillna("")

    fig_mag = px.scatter(
        mag_df,
        x="date",
        y="mag_value",
        color="category",
        color_discrete_map=cat_colors,
        size="mag_value",
        hover_name="title",
        hover_data={"mag_unit": True, "date": True},
        title="Event Magnitudes Over Time",
        labels={"mag_value": "Magnitude", "date": "Date"},
    )
    fig_mag.update_layout(template="plotly_dark", height=420)
    fig_mag.show()

    # Box plot of magnitudes by category
    fig_box = px.box(
        mag_df,
        x="category",
        y="mag_value",
        color="category",
        color_discrete_map=cat_colors,
        title="Magnitude Distribution by Category",
        points="outliers",
    )
    fig_box.update_layout(template="plotly_dark", height=400, showlegend=False, xaxis_title="")
    fig_box.show()
else:
    print("No magnitude data available in the current dataset.")

## 10 — Geographic hotspot heatmap

In [90]:
fig_density = px.density_map(
    df,
    lat="lat",
    lon="lon",
    radius=12,
    zoom=1,
    title="🔥 Global Hotspot Density of Natural Events",
    color_continuous_scale="Inferno",
    map_style="carto-darkmatter",
)
fig_density.update_layout(height=550, margin=dict(l=0, r=0, t=50, b=0))
fig_density.show()

## 11 — Day-of-week & hour-of-day patterns

In [91]:
df["dow"] = df["date"].dt.day_name()
df["hour"] = df["date"].dt.hour

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

heatmap_data = (
    df.groupby(["dow", "hour"])["id"]
    .nunique()
    .reset_index(name="events")
    .pivot(index="dow", columns="hour", values="events")
    .reindex(dow_order)
    .fillna(0)
)

fig_heat = px.imshow(
    heatmap_data,
    labels=dict(x="Hour (UTC)", y="Day of Week", color="Events"),
    title="Event Reporting Patterns: Day of Week × Hour (UTC)",
    color_continuous_scale="Plasma",
    aspect="auto",
)
fig_heat.update_layout(template="plotly_dark", height=350)
fig_heat.show()

---

# 🔥 Wildfire Deep Dive — When and Where Are Fires Burning?

Let's isolate wildfires from the EONET data and look at **when** they start, how long they burn, and seasonal patterns.

In [92]:
# Filter wildfires only
fires = df[df["category"] == "Wildfires"].copy()
fires_events = fires.drop_duplicates(subset="id")

n_fires = fires["id"].nunique()
n_fires_open = fires.loc[fires["is_open"], "id"].nunique()
fire_date_min = fires["date"].min()
fire_date_max = fires["date"].max()

fire_html = f"""
<div style="display:flex; gap:24px; flex-wrap:wrap; margin:12px 0;">
  <div style="background:#2a1a0e; color:#e0e0e0; padding:20px 28px; border-radius:12px; min-width:150px; text-align:center;">
    <div style="font-size:36px; font-weight:700; color:#ff4500;">{n_fires}</div>
    <div style="font-size:13px; margin-top:4px;">Total Wildfires</div>
  </div>
  <div style="background:#2a1a0e; color:#e0e0e0; padding:20px 28px; border-radius:12px; min-width:150px; text-align:center;">
    <div style="font-size:36px; font-weight:700; color:#ff6b6b;">{n_fires_open}</div>
    <div style="font-size:13px; margin-top:4px;">Still Burning</div>
  </div>
  <div style="background:#2a1a0e; color:#e0e0e0; padding:20px 28px; border-radius:12px; min-width:180px; text-align:center;">
    <div style="font-size:20px; font-weight:700; color:#ffd43b;">{fire_date_min:%b %d, %Y}</div>
    <div style="font-size:13px; margin-top:4px;">Earliest Fire</div>
  </div>
  <div style="background:#2a1a0e; color:#e0e0e0; padding:20px 28px; border-radius:12px; min-width:180px; text-align:center;">
    <div style="font-size:20px; font-weight:700; color:#ffd43b;">{fire_date_max:%b %d, %Y}</div>
    <div style="font-size:13px; margin-top:4px;">Most Recent Fire</div>
  </div>
</div>
"""
display(HTML(fire_html))
print(f"{len(fires)} data-points across {n_fires} unique wildfire events")

949 data-points across 949 unique wildfire events


### Wildfire timeline — when did each fire start and end?

Each bar shows a single wildfire event, spanning from its first to last reported geometry date. Still-burning fires are highlighted in red.

In [93]:
# Build a Gantt-style chart for wildfire events
fire_spans = (
    fires.groupby(["id", "title", "is_open"])
    .agg(start=("date", "min"), end=("date", "max"), data_points=("date", "count"))
    .reset_index()
)
# For open fires, extend the end to today
fire_spans.loc[fire_spans["is_open"], "end"] = pd.Timestamp.now(tz="UTC")
fire_spans["duration_days"] = (fire_spans["end"] - fire_spans["start"]).dt.days
fire_spans["status"] = fire_spans["is_open"].map({True: "🔴 Still Burning", False: "✅ Closed"})
fire_spans = fire_spans.sort_values("start")

# Show the top 40 by duration for readability
top_fires = fire_spans.nlargest(40, "duration_days")

fig_gantt = px.timeline(
    top_fires,
    x_start="start",
    x_end="end",
    y="title",
    color="status",
    color_discrete_map={"🔴 Still Burning": "#ff4500", "✅ Closed": "#6a9eda"},
    title="🔥 Wildfire Timelines — Top 40 Longest-Burning Fires",
    hover_data={"duration_days": True, "data_points": True, "start": True, "end": True},
)
fig_gantt.update_layout(
    template="plotly_dark",
    height=max(400, len(top_fires) * 18),
    yaxis=dict(categoryorder="array", categoryarray=top_fires.sort_values("start")["title"].tolist(), title=""),
    xaxis_title="",
    legend_title="Status",
)
fig_gantt.show()

### Wildfire seasonality — which months see the most fires?

In [94]:
# Monthly fire count
fires["month_name"] = fires["date"].dt.strftime("%b")
fires["month_num"] = fires["date"].dt.month

monthly_fires = (
    fires.drop_duplicates(subset=["id", "month_num"])
    .groupby(["month_num", "month_name"])["id"]
    .nunique()
    .reset_index(name="fires")
    .sort_values("month_num")
)

fig_season = px.bar(
    monthly_fires,
    x="month_name",
    y="fires",
    color="fires",
    color_continuous_scale="YlOrRd",
    title="🗓️ Wildfire Seasonality — Fires by Month",
    text="fires",
    labels={"fires": "Wildfires", "month_name": "Month"},
)
fig_season.update_layout(
    template="plotly_dark", height=380,
    xaxis=dict(categoryorder="array", categoryarray=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]),
    showlegend=False,
)
fig_season.update_traces(textposition="outside")
fig_season.show()

### Wildfire weekly activity & cumulative count

In [95]:
# Weekly new wildfire starts
fire_starts = fires.drop_duplicates(subset="id", keep="first").copy()
fire_starts["week"] = fire_starts["date"].dt.to_period("W").apply(lambda r: r.start_time)

weekly_starts = fire_starts.groupby("week")["id"].nunique().reset_index(name="new_fires")
weekly_starts = weekly_starts.sort_values("week")
weekly_starts["cumulative"] = weekly_starts["new_fires"].cumsum()

fig_weekly = make_subplots(specs=[[{"secondary_y": True}]])

fig_weekly.add_trace(
    go.Bar(
        x=weekly_starts["week"], y=weekly_starts["new_fires"],
        name="New Fires (weekly)",
        marker_color="#ff4500", opacity=0.7,
    ),
    secondary_y=False,
)
fig_weekly.add_trace(
    go.Scatter(
        x=weekly_starts["week"], y=weekly_starts["cumulative"],
        name="Cumulative Fires",
        line=dict(color="#ffd43b", width=3),
        mode="lines",
    ),
    secondary_y=True,
)
fig_weekly.update_layout(
    template="plotly_dark",
    title="🔥 Weekly New Wildfire Starts + Cumulative Total",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig_weekly.update_yaxes(title_text="New Fires per Week", secondary_y=False)
fig_weekly.update_yaxes(title_text="Cumulative Total", secondary_y=True)
fig_weekly.show()

/var/folders/8t/py7199996lzcf46ws17_9lp80000gn/T/ipykernel_30155/2476297528.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  fire_starts["week"] = fire_starts["date"].dt.to_period("W").apply(lambda r: r.start_time)


### Wildfire geographic map with timeline slider

In [96]:
# Animated scatter map — fires by month
fires_map = fires.drop_duplicates(subset=["id", "month"]).copy()
fires_map["month_str"] = fires_map["date"].dt.to_period("M").astype(str)
fires_map["status"] = fires_map["is_open"].map({True: "🔴 Burning", False: "✅ Closed"})

fig_fire_map = px.scatter_geo(
    fires_map.sort_values("month_str"),
    lat="lat",
    lon="lon",
    hover_name="title",
    hover_data={"date": True, "status": True, "lat": ":.2f", "lon": ":.2f"},
    animation_frame="month_str",
    color="status",
    color_discrete_map={"🔴 Burning": "#ff4500", "✅ Closed": "#6a9eda"},
    title="🗺️ Wildfire Locations by Month (use slider to scrub through time)",
    size_max=10,
)
fig_fire_map.update_traces(marker=dict(size=8, opacity=0.85, line=dict(width=0.4, color="white")))
fig_fire_map.update_geos(
    showcountries=True, countrycolor="#444",
    showcoastlines=True, coastlinecolor="#555",
    showland=True, landcolor="#1a1a2e",
    showocean=True, oceancolor="#0d1b2a",
    projection_type="natural earth",
)
fig_fire_map.update_layout(
    template="plotly_dark",
    height=550,
    margin=dict(l=0, r=0, t=50, b=0),
)
fig_fire_map.show()

/var/folders/8t/py7199996lzcf46ws17_9lp80000gn/T/ipykernel_30155/2878186548.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  fires_map["month_str"] = fires_map["date"].dt.to_period("M").astype(str)


### Wildfire event table with dates

In [97]:
# Detailed fire table with start/end dates
fire_table = fire_spans[[
    "title", "start", "end", "duration_days", "data_points", "status"
]].copy()
fire_table["start"] = fire_table["start"].dt.strftime("%Y-%m-%d")
fire_table["end"] = fire_table["end"].dt.strftime("%Y-%m-%d")
fire_table = fire_table.sort_values("duration_days", ascending=False).reset_index(drop=True)
fire_table.columns = ["Fire Name", "Start Date", "End Date", "Duration (days)", "Data Points", "Status"]

display(HTML(f"<h4>All {len(fire_table)} Wildfires — sorted by duration</h4>"))
fire_table.style.background_gradient(
    subset=["Duration (days)"], cmap="YlOrRd"
).set_properties(**{"text-align": "left"})

,Fire Name,Start Date,End Date,Duration (days),Data Points,Status
0,"RX Buff Creek Bill Prescribed Fire, Searcy, Arkansas",2026-02-27,2026-04-07,39,1,🔴 Still Burning
1,"RADAR RD (53) Wildfire, Polk, Florida",2026-02-27,2026-04-07,39,1,🔴 Still Burning
2,"RX PCS Square Rock 1 and 2 Prescribed Fire, Scott, Arkansas",2026-02-27,2026-04-07,39,1,🔴 Still Burning
3,"Rocky Branch RX Prescribed Fire, Texas, Missouri",2026-02-27,2026-04-07,39,1,🔴 Still Burning
4,"Mill Buzzard RX Prescribed Fire, Shannon, Missouri",2026-02-27,2026-04-07,39,1,🔴 Still Burning
5,"Pineknot South RX Prescribed Fire, Carter, Missouri",2026-02-27,2026-04-07,39,1,🔴 Still Burning
6,"RX CAT BU 36 Prescribed Fire, Grant, Louisiana",2026-02-27,2026-04-07,39,1,🔴 Still Burning
7,"Preusch-Smith RX Prescribed Fire, Pulaski, Missouri",2026-02-27,2026-04-07,39,1,🔴 Still Burning
8,"RX Pink Twist Prescribed Fire, Franklin, Arkansas",2026-02-27,2026-04-07,39,1,🔴 Still Burning
9,"RX Buffalo Danny Prescribed Fire, McCurtain, Oklahoma",2026-02-27,2026-04-07,39,1,🔴 Still Burning


---

# 🌡️ Climate Change Tracking

Beyond individual natural events, let's look at the **big picture** — how global temperatures and CO₂ concentrations have changed over time using data from **NASA GISS** and **NOAA**.

## 12 — Fetch global temperature & CO₂ data

- **Temperature**: NASA GISS Surface Temperature Analysis v4 (GISTEMP) — global land-ocean anomalies vs. 1951-1980 baseline, 1880–present.
- **CO₂**: NOAA Global Monitoring Lab — monthly mean CO₂ at Mauna Loa Observatory (Keeling Curve), 1958–present.

In [98]:
import io

# --- NASA GISS Global Temperature Anomalies (1880–present) ---
giss_url = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"
print("Fetching NASA GISS global temperature data…")
giss_resp = requests.get(giss_url, timeout=30)
giss_resp.raise_for_status()

# The CSV has a header row, then a second row with column labels; skip the first row
giss_df = pd.read_csv(io.StringIO(giss_resp.text), skiprows=1)

# Melt monthly columns into long format
month_cols = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
giss_long = giss_df[["Year"] + month_cols].melt(id_vars="Year", var_name="month_name", value_name="anomaly")
month_map = {m: i+1 for i, m in enumerate(month_cols)}
giss_long["month_num"] = giss_long["month_name"].map(month_map)
giss_long["anomaly"] = pd.to_numeric(giss_long["anomaly"], errors="coerce")
giss_long = giss_long.dropna(subset=["anomaly"])
giss_long["date"] = pd.to_datetime(giss_long["Year"].astype(str) + "-" + giss_long["month_num"].astype(str) + "-15")
giss_long = giss_long.sort_values("date").reset_index(drop=True)

# Annual averages
giss_annual = giss_long.groupby("Year")["anomaly"].mean().reset_index()
giss_annual = giss_annual[giss_annual["anomaly"].notna()]

print(f"✅ GISS: {len(giss_long)} monthly readings, {giss_annual['Year'].min()}–{giss_annual['Year'].max()}")

# --- NOAA CO₂ (Mauna Loa, 1958–present) ---
co2_url = "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv"
print("Fetching NOAA CO₂ data (Mauna Loa)…")
co2_resp = requests.get(co2_url, timeout=30)
co2_resp.raise_for_status()

# Skip comment lines starting with #
co2_lines = [l for l in co2_resp.text.splitlines() if not l.startswith("#")]
co2_df = pd.read_csv(io.StringIO("\n".join(co2_lines)))
co2_df.columns = co2_df.columns.str.strip()
co2_df = co2_df.rename(columns={"year": "year", "month": "month", "decimal date": "decimal_date", "average": "co2_ppm"})

# Some values are -99.99 (missing)
co2_df.loc[co2_df["co2_ppm"] < 0, "co2_ppm"] = float("nan")
co2_df = co2_df.dropna(subset=["co2_ppm"])
co2_df["date"] = pd.to_datetime(co2_df["year"].astype(str) + "-" + co2_df["month"].astype(str) + "-15")

# Annual averages
co2_annual = co2_df.groupby("year")["co2_ppm"].mean().reset_index()

print(f"✅ CO₂: {len(co2_df)} monthly readings, {co2_df['year'].min()}–{co2_df['year'].max()}")

Fetching NASA GISS global temperature data…
✅ GISS: 1754 monthly readings, 1880–2026
Fetching NOAA CO₂ data (Mauna Loa)…
✅ CO₂: 816 monthly readings, 1958–2026


## 13 — Global temperature anomaly (1880–present)

Temperature anomalies relative to the 1951–1980 average. The warming trend since the mid-20th century is unmistakable.

In [99]:
import numpy as np

# Monthly time series with smoothing
giss_long["anomaly_12m"] = giss_long["anomaly"].rolling(window=12, center=True).mean()

fig_temp = go.Figure()

# Monthly data (faint)
fig_temp.add_trace(go.Scatter(
    x=giss_long["date"], y=giss_long["anomaly"],
    mode="lines", name="Monthly",
    line=dict(color="rgba(100,149,237,0.25)", width=0.8),
))

# 12-month rolling average
fig_temp.add_trace(go.Scatter(
    x=giss_long["date"], y=giss_long["anomaly_12m"],
    mode="lines", name="12-Month Average",
    line=dict(color="#ff6b6b", width=2.5),
))

# Zero line (1951–1980 baseline)
fig_temp.add_hline(y=0, line_dash="dash", line_color="#888", annotation_text="1951–1980 baseline")

# Linear trend line
x_num = (giss_long["date"] - giss_long["date"].min()).dt.days.values.astype(float)
mask = ~np.isnan(giss_long["anomaly"].values)
z = np.polyfit(x_num[mask], giss_long["anomaly"].values[mask], 1)
trend = np.polyval(z, x_num)
fig_temp.add_trace(go.Scatter(
    x=giss_long["date"], y=trend,
    mode="lines", name="Linear Trend",
    line=dict(color="#ffd43b", width=1.5, dash="dot"),
))

fig_temp.update_layout(
    template="plotly_dark",
    title="🌡️ Global Temperature Anomaly (1880–Present) — NASA GISTEMP v4",
    yaxis_title="Temperature Anomaly (°C)",
    xaxis_title="",
    height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig_temp.show()

# Show latest reading
latest_year = giss_annual.iloc[-1]
print(f"Latest annual anomaly ({int(latest_year['Year'])}): +{latest_year['anomaly']:.2f}°C above 1951–1980 baseline")

Latest annual anomaly (2026): +1.16°C above 1951–1980 baseline


## 14 — Warming by decade

Average temperature anomaly for each decade since the 1880s — showing clear step-changes in warming.

In [100]:
# Decade averages
giss_annual["decade"] = (giss_annual["Year"] // 10 * 10).astype(int).astype(str) + "s"
decade_avg = giss_annual.groupby("decade")["anomaly"].mean().reset_index()
decade_avg = decade_avg.sort_values("decade")

# Color scale: blue (cold) → red (warm)
decade_avg["color"] = decade_avg["anomaly"]

fig_decade = px.bar(
    decade_avg,
    x="decade",
    y="anomaly",
    color="anomaly",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    title="Average Temperature Anomaly by Decade",
    text=decade_avg["anomaly"].apply(lambda x: f"{x:+.2f}°C"),
    labels={"anomaly": "Anomaly (°C)", "decade": "Decade"},
)
fig_decade.update_layout(template="plotly_dark", height=400, showlegend=False)
fig_decade.update_traces(textposition="outside")
fig_decade.show()

## 15 — CO₂ concentration: the Keeling Curve (1958–present)

Atmospheric CO₂ measured at Mauna Loa Observatory — one of the most important datasets in climate science.

In [101]:
fig_co2 = go.Figure()

fig_co2.add_trace(go.Scatter(
    x=co2_df["date"], y=co2_df["co2_ppm"],
    mode="lines", name="Monthly CO₂",
    line=dict(color="#51cf66", width=1.2),
    fill="tozeroy",
    fillcolor="rgba(81,207,102,0.1)",
))

# Annual average overlay
fig_co2.add_trace(go.Scatter(
    x=pd.to_datetime(co2_annual["year"].astype(str) + "-07-01"),
    y=co2_annual["co2_ppm"],
    mode="lines+markers", name="Annual Average",
    line=dict(color="#ffd43b", width=2.5),
    marker=dict(size=3),
))

# Key milestones
milestones = [
    (350, "350 ppm — 'Safe' limit (Hansen)"),
    (400, "400 ppm — First breached 2013"),
]
for ppm, label in milestones:
    if ppm <= co2_df["co2_ppm"].max() * 1.05:
        fig_co2.add_hline(y=ppm, line_dash="dash", line_color="#ff6b6b",
                          annotation_text=label, annotation_position="top left")

fig_co2.update_layout(
    template="plotly_dark",
    title="📈 Atmospheric CO₂ at Mauna Loa (Keeling Curve)",
    yaxis_title="CO₂ (ppm)",
    xaxis_title="",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig_co2.show()

latest_co2 = co2_df.iloc[-1]
print(f"Latest reading ({int(latest_co2['year'])}-{int(latest_co2['month']):02d}): {latest_co2['co2_ppm']:.1f} ppm")

Latest reading (2026-02): 429.4 ppm


## 16 — Temperature vs CO₂: the correlation

Plotting annual temperature anomaly against CO₂ concentration on a dual-axis chart reveals how closely the two track each other.

In [102]:
# Merge annual temperature and CO₂ data
merged = pd.merge(
    giss_annual.rename(columns={"Year": "year"}),
    co2_annual,
    on="year",
    how="inner",
)

fig_dual = make_subplots(specs=[[{"secondary_y": True}]])

fig_dual.add_trace(
    go.Bar(
        x=merged["year"], y=merged["anomaly"],
        name="Temp Anomaly (°C)",
        marker_color=merged["anomaly"].apply(lambda v: "#ff6b6b" if v > 0 else "#6a9eda"),
        opacity=0.7,
    ),
    secondary_y=False,
)

fig_dual.add_trace(
    go.Scatter(
        x=merged["year"], y=merged["co2_ppm"],
        name="CO₂ (ppm)",
        line=dict(color="#51cf66", width=3),
        mode="lines",
    ),
    secondary_y=True,
)

fig_dual.update_layout(
    template="plotly_dark",
    title="🔗 Temperature Anomaly & CO₂ Concentration (Annual)",
    height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig_dual.update_yaxes(title_text="Temperature Anomaly (°C)", secondary_y=False)
fig_dual.update_yaxes(title_text="CO₂ (ppm)", secondary_y=True)
fig_dual.show()

# Correlation
corr = merged[["anomaly", "co2_ppm"]].corr().iloc[0, 1]
print(f"Pearson correlation between annual temp anomaly and CO₂: {corr:.3f}")

Pearson correlation between annual temp anomaly and CO₂: 0.968


## 17 — Rate of warming: decade-over-decade change

How much faster is each decade warming compared to the last? This shows the *acceleration* of warming.

In [103]:
# Rate of warming: change from previous decade
decade_avg["prev_anomaly"] = decade_avg["anomaly"].shift(1)
decade_avg["change"] = decade_avg["anomaly"] - decade_avg["prev_anomaly"]
decade_rate = decade_avg.dropna(subset=["change"])

fig_rate = px.bar(
    decade_rate,
    x="decade",
    y="change",
    color="change",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    title="Decade-over-Decade Temperature Change (°C)",
    text=decade_rate["change"].apply(lambda x: f"{x:+.3f}"),
    labels={"change": "Change (°C)", "decade": "Decade"},
)
fig_rate.update_layout(template="plotly_dark", height=400, showlegend=False)
fig_rate.update_traces(textposition="outside")
fig_rate.show()

# Scatter: CO₂ vs Temperature
fig_scatter = px.scatter(
    merged,
    x="co2_ppm",
    y="anomaly",
    color="year",
    color_continuous_scale="Turbo",
    title="CO₂ vs Temperature Anomaly (each dot = one year)",
    labels={"co2_ppm": "CO₂ (ppm)", "anomaly": "Temp Anomaly (°C)", "year": "Year"},
    trendline="ols",
)
fig_scatter.update_layout(template="plotly_dark", height=420)
fig_scatter.update_traces(marker=dict(size=8, opacity=0.8))
fig_scatter.show()

---

**Data sources:**
- [NASA EONET v3 API](https://eonet.gsfc.nasa.gov/api/v3/events) · Earth Observatory Natural Event Tracker
- [NASA GISS GISTEMP v4](https://data.giss.nasa.gov/gistemp/) · Global Surface Temperature Analysis
- [NOAA GML](https://gml.noaa.gov/ccgg/trends/) · Mauna Loa CO₂ (Keeling Curve)

**Dashboard generated:** auto-refreshes on each run